# S&P 500 Correlation Network — Central vs Peripheral Portfolios

Builds a minimum spanning tree from the correlation structure of S&P 500 returns,
identifies **central** and **peripheral** stocks, then tests whether either group
beat the index out of sample.

**Method.** Correlations of log returns are converted to distances via the Mantegna
transform, `d = sqrt(2(1-rho))`. The MST keeps the `n-1` edges that connect every
stock using the tightest correlations, reducing ~125,000 pairwise numbers to a
readable tree. Central stocks sit at hubs; peripheral stocks sit far out on the
branches.

**Design.** The tree is built on a *formation* window (Jan 2024 - Jun 2026). The
portfolios are then held, untouched, over a *test* window (Jul 2026 - present).
Selecting and testing on the same data would guarantee a flattering result.

**Known limitations**, stated up front:

- Index membership is scraped as of today, so companies dropped since 2024 are
  missing and recent additions are backfilled. This is survivorship bias.
- The test window is short. Treat the outcome as illustrative, not as evidence
  that either portfolio construction works.

## 1. Setup

In [ ]:
import io
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

In [ ]:
# ---- Configuration -------------------------------------------------------

FORMATION_START = "2024-01-01"     # window the network is built on
FORMATION_END   = "2026-07-01"     # exclusive

TEST_START      = "2026-07-01"     # window the portfolios are tested on
TEST_END        = pd.Timestamp.today().normalize()

INDEX_TICKER    = "^GSPC"          # the actual S&P 500 index
PORTFOLIO_SIZE  = 15               # stocks per portfolio
INITIAL_CAPITAL = 100_000

BATCH, PAUSE    = 50, 2            # download batching, seconds between batches
CACHE           = Path("cache")    # downloaded data is cached here
CACHE.mkdir(exist_ok=True)

WIKI = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Roll back to the most recent Friday, so the window ends on a full trading week
TEST_END = TEST_END - pd.Timedelta(days=(TEST_END.weekday() - 4) % 7)

print(f"Formation: {FORMATION_START} -> {FORMATION_END}")
print(f"Test:      {TEST_START} -> {TEST_END.date()} (last Friday)")

## 2. Data

In [ ]:
def get_constituents():
    """Current S&P 500 membership from Wikipedia -> ticker, company, sector.

    Wikipedia rejects urllib's default user-agent with a 403, so the page is
    fetched with requests and handed to pandas as a string.
    """
    cached = CACHE / "constituents.csv"
    if cached.exists():
        return pd.read_csv(cached)

    r = requests.get(WIKI, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    table = pd.read_html(io.StringIO(r.text))[0]

    df = table.rename(columns={"Symbol": "ticker", "Security": "company",
                               "GICS Sector": "sector"})[["ticker", "company", "sector"]]
    # Yahoo writes class shares with a hyphen: BRK.B -> BRK-B
    df["ticker"] = df["ticker"].str.strip().str.replace(".", "-", regex=False)
    df = df.drop_duplicates("ticker").reset_index(drop=True)

    df.to_csv(cached, index=False)
    return df


members = get_constituents()
sectors = members.set_index("ticker")["sector"]
print(f"{len(members)} constituents")
members.head()

In [ ]:
def download_closes(tickers, start, end, interval, tag):
    """Adjusted closes for many tickers -> wide frame (dates x tickers)."""
    cached = CACHE / f"px_{tag}.csv"
    if cached.exists():
        return pd.read_csv(cached, index_col="Date", parse_dates=True)

    frames, failed = [], []
    for i in range(0, len(tickers), BATCH):
        chunk = tickers[i:i + BATCH]
        print(f"  batch {i // BATCH + 1}/{-(-len(tickers) // BATCH)}", file=sys.stderr)
        try:
            data = yf.download(chunk, start=start, end=end, interval=interval,
                               auto_adjust=True, progress=False, threads=True)
        except Exception as e:
            print(f"  failed: {e}", file=sys.stderr)
            failed.extend(chunk)
            continue

        if data.empty:
            failed.extend(chunk)
            continue

        closes = data["Close"] if isinstance(data.columns, pd.MultiIndex) \
                 else data[["Close"]].rename(columns={"Close": chunk[0]})

        empty = [c for c in closes.columns if closes[c].isna().all()]
        failed.extend(empty)
        frames.append(closes.drop(columns=empty))
        time.sleep(PAUSE)

    wide = pd.concat(frames, axis=1).sort_index().round(2)
    wide.index.name = "Date"
    wide.to_csv(cached)
    if failed:
        print(f"No data for {len(failed)}: {', '.join(sorted(set(failed))[:20])}")
    return wide

In [ ]:
tickers = members["ticker"].tolist()

# Formation: weekly is enough to estimate correlation structure over 2.5 years
prices_formation = download_closes(tickers, FORMATION_START, FORMATION_END, "1wk", "formation")

# Test: daily, so the equity curves have enough resolution to be readable
prices_test = download_closes(tickers, TEST_START, TEST_END, "1d", "test")

print("formation:", prices_formation.shape, " test:", prices_test.shape)

In [ ]:
# The index itself. This is what the earlier version was missing -- slicing the
# constituent matrix gives you 500 stocks, not the S&P 500.
def download_index(start, end, interval, tag):
    cached = CACHE / f"idx_{tag}.csv"
    if cached.exists():
        s = pd.read_csv(cached, index_col="Date", parse_dates=True)["close"]
        return s

    data = yf.download(INDEX_TICKER, start=start, end=end, interval=interval,
                       auto_adjust=True, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    s = data["Close"].rename("close")
    s.index.name = "Date"
    s.to_frame().to_csv(cached)
    return s


sp500_test = download_index(TEST_START, TEST_END, "1d", "test")
print(f"S&P 500: {len(sp500_test)} daily observations, "
      f"{sp500_test.index[0].date()} -> {sp500_test.index[-1].date()}")
sp500_test.head()

### Missing data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.heatmap(prices_formation.T.isnull(), ax=axes[0], cbar=False)
axes[0].set_title("Formation window — missing values")
sns.heatmap(prices_test.T.isnull(), ax=axes[1], cbar=False)
axes[1].set_title("Test window — missing values")
plt.tight_layout()

## 3. Returns

Two corrections from the original notebook.

The sign was inverted: `log(P.shift(1)) - log(P)` gives the negative of the return.
Correlation is invariant to a sign flip applied to both series, so the MST was
unaffected, but the returns themselves were wrong.

Cleaning now happens per window and per year. A ticker with an all-NaN column in
one year produces `NaN` correlations, and `nx.minimum_spanning_tree` cannot rank
`NaN` edge weights — that was the `ValueError: NaN found as an edge weight`.

In [ ]:
def log_returns(prices):
    """Log returns. Note the order: log(P_t) - log(P_{t-1})."""
    return np.log(prices) - np.log(prices.shift(1))


def clean(returns, min_frac=0.8):
    """Drop sparse and constant columns, then any remaining ragged rows."""
    r = returns.dropna(axis=1, thresh=int(min_frac * len(returns)))
    r = r.loc[:, r.std() > 0]                 # zero variance -> NaN correlation
    return r.dropna(axis=0, how="any")


rets_formation = clean(log_returns(prices_formation))
print(f"{rets_formation.shape[1]} tickers x {rets_formation.shape[0]} weekly returns")

In [ ]:
# Per-year slices, restricted to tickers present in all of them so the three
# trees have identical node sets and are actually comparable.
years = [2024, 2025, 2026]
yearly = {y: clean(log_returns(prices_formation.loc[str(y)])) for y in years}

common = yearly[2024].columns
for y in years[1:]:
    common = common.intersection(yearly[y].columns)
yearly = {y: df[common] for y, df in yearly.items()}

for y in years:
    print(f"{y}: {yearly[y].shape[0]:>3} observations, {yearly[y].shape[1]} tickers")

print("\nNote: 2026 is a half year, so its correlations rest on far fewer\n"
      "observations and its tree will look noisier for reasons unrelated to markets.")

## 4. Correlation and distance

In [ ]:
def to_distance(corr):
    """Mantegna distance. The clip guards against floating point pushing
    1 - rho a hair below zero, which would produce NaN from the sqrt."""
    return np.sqrt(np.clip(2 * (1 - corr), 0, None))


corr_full = rets_formation.corr()
dist_full = to_distance(corr_full)
assert corr_full.notna().all().all(), "NaN in correlation matrix"

corr_yearly = {y: yearly[y].corr() for y in years}
dist_yearly = {y: to_distance(c) for y, c in corr_yearly.items()}

print(f"Mean off-diagonal correlation, full window: "
      f"{corr_full.values[np.triu_indices_from(corr_full, k=1)].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
for ax, y in zip(axes, years):
    sns.heatmap(corr_yearly[y], ax=ax, cmap="coolwarm", center=0, vmin=-0.4, vmax=1,
                xticklabels=False, yticklabels=False, cbar_kws={"shrink": 0.7})
    ax.set_title(f"{y} return correlations")
plt.tight_layout()

## 5. Minimum spanning trees

In [ ]:
def to_mst(dist):
    """Distance matrix -> MST. Self-loops from the zero diagonal must go first,
    otherwise they pollute the edge set."""
    G = nx.Graph(dist)
    G.remove_edges_from(nx.selfloop_edges(G))
    T = nx.minimum_spanning_tree(G)
    assert T.number_of_edges() == T.number_of_nodes() - 1, "not a tree"
    assert nx.is_connected(T), "disconnected — got a forest, not a tree"
    return T


mst_full = to_mst(dist_full)
mst_yearly = {y: to_mst(dist_yearly[y]) for y in years}

print(f"Full-window MST: {mst_full.number_of_nodes()} nodes, "
      f"{mst_full.number_of_edges()} edges, connected={nx.is_connected(mst_full)}")

### Layout

`draw_kamada_kawai` was producing a ring with everything crossing the middle.
Kamada-Kawai minimises a stress function over all node pairs and degenerates at
500 nodes. A spring layout with tuned repulsion handles trees this size; colouring
by GICS sector is what makes the clustering legible.

In [ ]:
def draw_tree(T, title, color_by="sector", highlight=None, figsize=(20, 20),
              label_hubs=6, seed=42):
    pos = nx.spring_layout(T, k=0.35, iterations=300, seed=seed)
    deg = dict(T.degree())

    if color_by == "sector":
        cats = sorted(sectors.reindex(list(T.nodes())).dropna().unique())
        palette = dict(zip(cats, plt.cm.tab20.colors))
        colors = [palette.get(sectors.get(n), "lightgray") for n in T.nodes()]
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=s)
                   for s, c in palette.items()]
    else:
        central, peripheral = highlight
        colors = ["#d62728" if n in central else
                  "#2ca02c" if n in peripheral else "#c7d0d9" for n in T.nodes()]
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=l)
                   for c, l in [("#d62728", "Central"), ("#2ca02c", "Peripheral"),
                                ("#c7d0d9", "Other")]]

    plt.figure(figsize=figsize)
    nx.draw_networkx_edges(T, pos, alpha=0.3, width=0.6)
    nx.draw_networkx_nodes(T, pos, node_color=colors, linewidths=0,
                           node_size=[20 + 12 * deg[n] for n in T.nodes()])

    to_label = ([n for n, d in deg.items() if d >= label_hubs] if highlight is None
                else list(highlight[0]) + list(highlight[1]))
    nx.draw_networkx_labels(T, pos, labels={n: n for n in to_label}, font_size=9)

    plt.legend(handles=handles, loc="upper left", frameon=True, fontsize=9)
    plt.title(title, fontsize=15)
    plt.axis("off")
    plt.tight_layout()


draw_tree(mst_full, "S&P 500 correlation MST, Jan 2024 – Jun 2026 (coloured by sector)")

## 6. Network statistics over time

In [ ]:
stats = pd.DataFrame({
    "year": years,
    "avg_shortest_path": [nx.average_shortest_path_length(mst_yearly[y]) for y in years],
    "diameter":          [nx.diameter(mst_yearly[y]) for y in years],
    "max_degree":        [max(dict(mst_yearly[y].degree()).values()) for y in years],
    "observations":      [len(yearly[y]) for y in years],
})

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col in zip(axes, ["avg_shortest_path", "diameter", "max_degree"]):
    sns.lineplot(data=stats, x="year", y=col, marker="o", ax=ax)
    ax.set_xticks(years)
    ax.set_title(col.replace("_", " "))
plt.tight_layout()
stats

A shrinking average path length means the market is tightening — stocks moving
more in lockstep, which typically happens in stressed periods. Read the 2026
figure with care given it rests on half a year of data.

## 7. Centrality and portfolio selection

**Centrality** is averaged over degree and betweenness. The original averaged the
raw values, but betweenness ranges up to ~0.76 here while degree tops out near
0.11, so the mean was effectively betweenness alone. Averaging *percentile ranks*
gives both measures equal say.

**Peripherality** keeps the three criteria from the original: hop distance from
the highest-degree node, from the most-correlated node, and mean hop distance to
all other nodes.

In [ ]:
degree_c      = nx.degree_centrality(mst_full)
betweenness_c = nx.betweenness_centrality(mst_full)
closeness_c   = nx.closeness_centrality(mst_full)
eigenvector_c = nx.eigenvector_centrality_numpy(mst_full)

node_stats = pd.DataFrame({
    "degree_centrality":      degree_c,
    "betweenness_centrality": betweenness_c,
    "closeness_centrality":   closeness_c,
    "eigenvector_centrality": eigenvector_c,
})

# Rank-average, so the two measures contribute equally
node_stats["centrality_score"] = (
    node_stats["degree_centrality"].rank(pct=True) +
    node_stats["betweenness_centrality"].rank(pct=True)
) / 2

node_stats.sort_values("centrality_score", ascending=False).head(10)

In [ ]:
# --- Peripherality: three hop-distance criteria ---------------------------
all_pairs = dict(nx.all_pairs_shortest_path_length(mst_full))
nodes = list(mst_full.nodes())

hub_degree = max(degree_c, key=degree_c.get)

neighbour_corr = {n: sum(corr_full.loc[n, nb] for nb in mst_full.neighbors(n)) for n in nodes}
hub_corr = max(neighbour_corr, key=neighbour_corr.get)

node_stats["dist_from_degree_hub"] = [all_pairs[n][hub_degree] for n in nodes]
node_stats["dist_from_corr_hub"]   = [all_pairs[n][hub_corr] for n in nodes]
node_stats["mean_dist_to_all"]     = [
    np.mean([d for m, d in all_pairs[n].items() if m != n]) for n in nodes
]

node_stats["peripherality_score"] = node_stats[
    ["dist_from_degree_hub", "dist_from_corr_hub", "mean_dist_to_all"]
].rank(pct=True).mean(axis=1)

print(f"Highest-degree hub: {hub_degree}   Most-correlated hub: {hub_corr}")
node_stats.sort_values("peripherality_score", ascending=False).head(10)

In [ ]:
tradable = [t for t in node_stats.index if t in prices_test.columns
            and prices_test[t].notna().all()]

central_portfolio = (node_stats.loc[tradable]
                     .sort_values("centrality_score", ascending=False)
                     .head(PORTFOLIO_SIZE).index.tolist())

peripheral_portfolio = (node_stats.loc[tradable]
                        .sort_values("peripherality_score", ascending=False)
                        .head(PORTFOLIO_SIZE).index.tolist())

overlap = set(central_portfolio) & set(peripheral_portfolio)
assert not overlap, f"portfolios overlap: {overlap}"

pd.DataFrame({
    "Central": [f"{t} ({sectors.get(t, '?')})" for t in central_portfolio],
    "Peripheral": [f"{t} ({sectors.get(t, '?')})" for t in peripheral_portfolio],
})

In [ ]:
draw_tree(mst_full, "Central (red) vs peripheral (green) stocks",
          color_by="highlight", highlight=(set(central_portfolio),
                                           set(peripheral_portfolio)))

## 8. Out-of-sample comparison

Portfolios are **equal-weighted**: capital is split evenly across the 15 names and
share counts are fixed at the first trading day. The original summed raw prices,
which silently weights by share price — a \$500 stock carried ten times the weight
of a \$50 one for no economic reason.

All three series start at the same capital so the curves are directly comparable.

In [ ]:
def equal_weight_curve(prices, tickers, capital=INITIAL_CAPITAL):
    """Buy and hold, equal capital per name, shares fixed at inception."""
    px = prices[tickers].dropna()
    shares = (capital / len(tickers)) / px.iloc[0]
    return (px * shares).sum(axis=1)


central_curve    = equal_weight_curve(prices_test, central_portfolio)
peripheral_curve = equal_weight_curve(prices_test, peripheral_portfolio)
index_curve      = sp500_test / sp500_test.iloc[0] * INITIAL_CAPITAL

performance = pd.DataFrame({
    "S&P 500":    index_curve,
    "Central":    central_curve,
    "Peripheral": peripheral_curve,
}).dropna()

print(f"{len(performance)} trading days, "
      f"{performance.index[0].date()} -> {performance.index[-1].date()}")
performance.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
styles = {"S&P 500": ("black", "--", 2.0),
          "Central": ("#d62728", "-", 1.8),
          "Peripheral": ("#2ca02c", "-", 1.8)}

for col, (c, ls, lw) in styles.items():
    ax.plot(performance.index, performance[col], label=col, color=c, ls=ls, lw=lw)

ax.axhline(INITIAL_CAPITAL, color="gray", lw=0.8, alpha=0.5)
ax.set_title(f"Equal-weighted MST portfolios vs the S&P 500\n"
             f"{performance.index[0].date()} to {performance.index[-1].date()}, "
             f"${INITIAL_CAPITAL:,} initial", fontsize=13)
ax.set_ylabel("Portfolio value ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.legend(loc="upper left")
plt.tight_layout()

In [ ]:
def summarise(curve):
    r = curve.pct_change().dropna()
    total = curve.iloc[-1] / curve.iloc[0] - 1
    years_held = (curve.index[-1] - curve.index[0]).days / 365.25
    drawdown = (curve / curve.cummax() - 1).min()
    vol = r.std() * np.sqrt(252)
    return pd.Series({
        "Total return":      total,
        "Annualised return": (1 + total) ** (1 / years_held) - 1 if years_held > 0 else np.nan,
        "Annualised vol":    vol,
        "Sharpe (rf=0)":     (r.mean() * 252) / vol if vol > 0 else np.nan,
        "Max drawdown":      drawdown,
        "Final value":       curve.iloc[-1],
    })


summary = performance.apply(summarise).T
display(summary.style.format({
    "Total return": "{:.2%}", "Annualised return": "{:.2%}",
    "Annualised vol": "{:.2%}", "Sharpe (rf=0)": "{:.2f}",
    "Max drawdown": "{:.2%}", "Final value": "${:,.0f}",
}))

In [ ]:
# Excess return over the index, to see when any gap actually opened up
excess = performance[["Central", "Peripheral"]].div(performance["S&P 500"], axis=0) - 1

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(excess.index, excess["Central"], color="#d62728", label="Central − S&P 500")
ax.plot(excess.index, excess["Peripheral"], color="#2ca02c", label="Peripheral − S&P 500")
ax.axhline(0, color="black", lw=1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
ax.set_title("Cumulative relative performance versus the index")
ax.legend()
plt.tight_layout()

In [ ]:
n_obs = len(performance)
print(f"Test window: {n_obs} trading days "
      f"({(performance.index[-1] - performance.index[0]).days} calendar days)\n")
print("How much to read into this:\n")
print(f"  - {n_obs} observations is far too few to separate skill from noise.")
print("    Differences of a few percent over this span are well within what")
print("    random selection of 15 stocks would produce.")
print("  - The universe is today's index membership, so anything delisted or")
print("    removed since 2024 is absent. That bias favours the portfolios.")
print("  - Returns are gross: no transaction costs, no slippage, no taxes.")
print("  - One window, one starting date. A rolling-window study — re-forming")
print("    the tree every quarter and chaining the results — is what would")
print("    actually test the idea.")

## 9. Where to take this

The single most valuable extension is **rolling windows**: rebuild the MST every
quarter on a trailing year of returns, reselect both portfolios, and chain the
out-of-sample periods. That turns one noisy observation into a time series and is
the difference between an anecdote and a finding.

Beyond that:

- **Point-in-time membership.** Wikipedia's second table lists index additions and
  removals with dates; walking it backwards reconstructs the true universe and
  removes the survivorship bias.
- **Compare against a fair benchmark.** These portfolios are equal-weighted while
  the S&P 500 is cap-weighted, so part of any gap is just the equal-weight effect.
  RSP, the equal-weight index ETF, is the cleaner comparison.
- **Risk, not just return.** The literature's stronger claim about peripheral
  stocks concerns diversification benefit, not outperformance. Comparing realised
  portfolio volatility is more likely to show something real than comparing returns.